# Pronóstico de demanda · Equipo 43
**Proyecto Integrador · Maestría en Inteligencia Artificial Aplicada (MNA), Tecnológico de Monterrey**

Daniela Juárez · Kevin Valdez · Yael Márquez
Asesora: Dra. Eduviges Ludivina Facundo · Patrocinador: Dr. Juan Arturo Nolazco Flores · Empresa: Bnext 

**Objetivo.** Pronosticar la venta diaria por tienda y categoría de 1 a 14 días hacia adelante, con un rango
(P10–P50–P90) y no sólo un número, y operar el modelo con MLOps en Databricks: datos versionados, validación
automática, registro de modelos, compuerta de calidad, inferencia programada y monitoreo.

**¿Cómo vamos?** Abre el notebook **`ESTADO_DEL_PROYECTO`** de esta carpeta: muestra qué etapas se han corrido,
las cifras clave y los hallazgos más recientes.

## Organización de esta carpeta

```
/Workspace/Shared/mna_equipo43/
├── LEEME                      ← esta portada
├── ESTADO_DEL_PROYECTO        ← avance, cifras clave y hallazgos (sólo lee)
├── 1_datos/                   ← fase 1: entender, auditar y limpiar los datos
│   ├── 00_entorno_y_permisos
│   ├── 01_inventario_catalogo
│   ├── 02_carga_bronce
│   ├── 03_auditoria_trabajo_previo
│   ├── 04_eda_calidad
│   ├── 05_eda_tiempo_y_tiendas
│   ├── 06_eda_variables_y_promociones
│   ├── 07_eda_externas
│   ├── 08_limpieza_plata
│   ├── 09_panel_diario
│   ├── 10_eda_series_y_linea_base
│   ├── 11_validacion_calidad     ← compuerta de datos (con DETENER_SI_FALLA = True en el Job)
│   ├── 12_evidencia_calidad      ← parquets para Bnext en el volumen maestria.hallazgos_calidad
│   └── 13_calendario_promociones_plan ← Excel editable del volumen → tablas del plan de promociones (futuras + histórico)
├── 2_modelos/                 ← fase 2: variables por origen y modelos
│   ├── 20_features_oro           ← tabla Oro (serie × origen × horizonte) con prueba anti-fuga
│   ├── 21_avance4_modelos_alternativos ← Avance 4: torneo de modelos individuales, ajuste fino y modelo final
│   └── Avance5.43                ← Avance 5: ajuste fino, ensambles, prueba no vista (septiembre), intervalos, compuerta, registro y pronóstico
└── 3_operacion/               ← fase 3 (Avance 6): operación con MLOps y productos para el negocio
    ├── 22_pronostico_operativo   ← inferencia con el modelo registrado (sin reentrenar) → gold.pronostico_14d
    ├── 23_monitoreo              ← acierto en producción, deriva (PSI), frescura y alertas
    ├── 24_promocion_modelo       ← @challenger → @champion con evidencia y aprobaciones de negocio
    ├── 30_simulacion_escenarios_whatif ← escenarios de promoción con el modelo (+ uplift histórico)
    ├── 31_desagregacion_predicciones_sku ← reparto a producto con fila OTROS y techo de demanda en unidades
    └── 40_salidas_dashboard      ← tablas gold.dash_* que lee el tablero AI/BI
```

Cada notebook trae su código completo (no depende de otros archivos), explica arriba qué preguntas responde, qué lee
y qué escribe, y en cada sección dice **qué hace** y **qué mirar**. Todos terminan con un **RESUMEN PARA REVISIÓN**
y guardan sus hallazgos en `mlops.hallazgos`. Los notebooks conservan sus resultados: se pueden consultar sin volver a correrlos.

## Datos

**Fuente (Bnext, sólo lectura).** Catálogo `modeladoprescriptivoventas`, esquema `maestria`:

| Volumen | Contenido |
| --- | --- |
| `/Volumes/modeladoprescriptivoventas/maestria/2026-03` … `2026-09` | Líneas de venta de cada mes (muchos parquet por mes) |
| `/Volumes/modeladoprescriptivoventas/maestria/processed` | Fuentes externas (tiendas, códigos postales, INEGI, clima, calendarios) y el ABT del equipo anterior |
| `/Volumes/modeladoprescriptivoventas/maestria/outputs` | Modelo campeón heredado y sus parámetros |

**Tablas del equipo** (arquitectura medallón). Viven en `modeladoprescriptivoventas.maestria` con el prefijo de su capa,
porque el equipo puede crear tablas, modelos y volúmenes en `maestria` pero no esquemas nuevos:

| Capa | Tabla en `maestria` | Qué es | La crea |
| --- | --- | --- | --- |
| Bronce | `bronze_ventas_lineas` | Todos los meses tal como llegan, con el archivo de origen de cada fila | `02` |
| Plata | `silver_ventas_lineas` | Líneas limpias, únicas y homologadas | `08` |
| Plata | `silver_catalogo_categorias`, `silver_catalogo_marcas`, `silver_catalogo_tiendas` | Dimensiones homologadas | `08` |
| Plata | `silver_panel_categoria_dia` | Una fila por tienda, categoría y día, con relleno acotado a la ventana de la tienda, días sin datos y atípicos marcados, tipo de categoría y promociones vigentes | `09` |
| Plata | `silver_promociones`, `silver_calendario_promociones` | Vigencia de cada promoción (inicio, fin, censura) y sus tiendas y categorías | `09` |
| Plata | `silver_calendario_promociones_plan`, `silver_calendario_promociones_plan_dia` | Plan de promociones (archivo editable + histórico observado) por promoción y por tienda × categoría × día | `13` |
| Oro | `gold_features_origen` | Variables para el modelo (serie × origen × horizonte) | `20` |
| Oro | `gold_pronostico_14d`, `gold_predicciones_sku`, `gold_surtido_sku_semana`, `gold_escenarios_whatif*`, `gold_uplift_promocional_historico`, `gold_dash_*` | Pronóstico operativo, reparto a producto, escenarios what-if y tablas del tablero | `22`, `31`, `30`, `40` |
| MLOps | `mlops_av5_*`, `mlops_monitoreo_modelo`, `mlops_monitoreo_horizonte`, `mlops_deriva_variables`, `mlops_alertas`, `mlops_aprobaciones`, `mlops_registro_modelos`, `mlops_pronosticos_bitacora`, `mlops_corridas` | Resultados del Avance 5, monitoreo, alertas, aprobaciones y bitácoras de operación | `Avance5.43`, `22`–`24` |
| MLOps | `mlops_bronce_cargas`, `mlops_conciliacion_plata`, `mlops_validaciones` | Bitácoras de carga, conciliación Bronce → Plata y contratos de calidad | `02`, `08`, `11` |
| MLOps | `mlops_eda_*`, `mlops_auditoria_previo`, `mlops_linea_base_ingenua*`, `mlops_comparacion_modelo_heredado`, `mlops_hallazgos` | Exploración, auditoría, línea base y hallazgos | `03`–`10` |
| Volumen | `/Volumes/modeladoprescriptivoventas/maestria/hallazgos_calidad/` | Parquets con las líneas detrás de los hallazgos de calidad (importes imposibles, duplicados, venta en cero, % de descuento) y su `LEEME.md`, para que Bnext los revise | `12` |
| Modelos | `maestria.demanda_categoria` (registro de Unity Catalog) | Modelo final con alias `challenger` (lo registra `Avance5.43`) y `champion` (lo promueve `24`) | `Avance5.43`, `24` |

En el texto de cada notebook las tablas se nombran por capa (`bronze.ventas_lineas`); el nombre real es el de esta tabla.

## Fases del proyecto

| Fase | Qué incluye | Estado |
| --- | --- | --- |
| **1 · Datos** | Inventario, carga de los meses, auditoría del trabajo previo, EDA, limpieza, panel diario, línea base ingenua y contratos de calidad | En ejecución |
| **2 · Modelos** | Variables sin fuga para h = 1…14, línea base LightGBM reentrenada con todos los meses, torneo de pérdidas (Tweedie, L1, Poisson), cuantiles P10/P90 calibrados, validación walk-forward | Siguiente |
| **3 · MLOps** | MLflow y registro en Unity Catalog (`champion` / `challenger`), compuerta de calidad, inferencia a 14 días, monitoreo de deriva y error, jobs programados | Después |
| **4 · Consumo** | Tablero AI/BI y escenarios de precio y promoción (what-if) | Si alcanza el tiempo |

**Trabajo previo.** El equipo anterior (modelo LightGBM, marzo–mayo, WMAPE 0.5002) y los Avances 2 y 3 se hicieron en
el servidor de Bnext y en computadoras locales. Aquí se usan sólo como referencia: el notebook `03` verifica sus cifras
contra los datos de Databricks y todo lo demás se rehace aquí con todos los meses.

## Cómo correr la fase 1
1. Cómputo: **serverless** (el que usa el equipo) o un clúster con Unity Catalog.
2. Abrir `1_datos/00_entorno_y_permisos` → **Run all**: confirma permisos, volúmenes y librerías.
3. Correr en orden del `01` al `11`. Cada uno tarda de 5 a 60 minutos; el `02` y el `08` son los más pesados.
4. Abrir `ESTADO_DEL_PROYECTO` → **Run all** para ver el avance.

**Configuración.** La primera celda de código de cada notebook es idéntica en todos: catálogo, meses, esquemas y la
columna que define el día de la venta. Si algo cambia (por ejemplo, llega un mes nuevo), se cambia ahí en todos.

**Privacidad (NDA).** Los notebooks sólo imprimen totales, porcentajes y distribuciones; nunca registros individuales
ni ubicaciones de tiendas.